# JSON in SQL: Basics

## Overview
Modern SQL databases support storing and querying JSON natively, blending relational and document models. This notebook covers the fundamentals using SQLite's `json_extract()` and `json_each()`, with PostgreSQL JSONB equivalents shown throughout.

**SQLite vs PostgreSQL JSON support:**

| Feature | SQLite | PostgreSQL |
|---|---|---|
| JSON type | `JSON` (TEXT alias) | `JSON` (text) or `JSONB` (binary, indexed) |
| Read field | `json_extract(col, '$.key')` | `col->>'key'` |
| Read nested | `json_extract(col, '$.a.b')` | `col->'a'->>'b'` |
| Array element | `json_extract(col, '$.arr[0]')` | `col->'arr'->0` |
| Expand array | `json_each(col)` | `jsonb_array_elements(col)` |
| Build object | `json_object(k,v,...)` | `json_build_object(k,v,...)` |
| Containment | Not built-in | `col @> '{"key":"val"}'::jsonb` |
| Expression index | `CREATE INDEX ON t (json_extract(col,'$.key'))` | `CREATE INDEX ON t ((col->>'key'))` |

**Dataset:** 5 patients with nested JSON `demographics` (age, province, contact, insurance), `clinical` (conditions array, allergies, vitals history), and `preferences` objects.

---

In [ ]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


---
## json_extract: reading scalar values

In [ ]:
print("=== json_extract: reading values from JSON columns ===")
print()

# Basic json_extract
rows = conn.execute("""
    SELECT full_name,
           json_extract(demographics, '$.age')      AS age,
           json_extract(demographics, '$.province') AS province,
           json_extract(demographics, '$.contact.phone') AS phone
    FROM   patients
    ORDER  BY full_name
""").fetchall()
print("Basic json_extract:")
print(f"  {'full_name':<22s}  {'age':>4s}  {'province':<10s}  phone")
print("  " + "-"*55)
for r in rows:
    print(f"  {r['full_name']:<22s}  {r['age']:>4d}  {r['province']:<10s}  {r['phone']}")

print()
# Nested path
rows2 = conn.execute("""
    SELECT full_name,
           json_extract(demographics, '$.insurance.plan') AS plan,
           json_extract(demographics, '$.insurance.id')   AS ins_id
    FROM   patients
    ORDER  BY full_name
""").fetchall()
print("Nested path $.insurance.plan:")
for r in rows2:
    print(f"  {r['full_name']:<22s}  {r['plan']:<10s}  {r['ins_id']}")

print()
print("PostgreSQL JSONB equivalents:")
pg_equiv = [
    ("json_extract(col,'$.key')",       "col->>'key'           -- returns text"),
    ("json_extract(col,'$.a.b')",       "col->'a'->>'b'        -- chained arrow operators"),
    ("json_extract(col,'$.arr[0]')",    "col->'arr'->0         -- array element (integer index)"),
    ("json_extract(col,'$.arr[0].x')", "col->'arr'->0->>'x'   -- nested into array element"),
    ("WHERE json_extract = 'NB'",       "WHERE col->>'province' = 'NB'"),
    ("WHERE json_extract > 50",         "WHERE (col->>'age')::int > 50"),
]
print(f"  {'SQLite':<42s}  PostgreSQL JSONB")
print("  " + "-"*75)
for sqlite_syn, pg_syn in pg_equiv:
    print(f"  {sqlite_syn:<42s}  {pg_syn}")


---
## Filtering on JSON values

In [ ]:
print("=== Filtering on JSON values ===")
print()

# WHERE on json_extract
rows = conn.execute("""
    SELECT full_name,
           json_extract(demographics, '$.province') AS province,
           json_extract(demographics, '$.age')      AS age
    FROM   patients
    WHERE  json_extract(demographics, '$.province') IN ('ON','BC')
    ORDER  BY full_name
""").fetchall()
print("Patients in ON or BC:")
for r in rows:
    print(f"  {r['full_name']:<22s}  {r['province']}  age={r['age']}")

print()
# Filter on nested boolean
rows2 = conn.execute("""
    SELECT full_name,
           json_extract(clinical, '$.smoker') AS smoker
    FROM   patients
    WHERE  json_extract(clinical, '$.smoker') = 1
""").fetchall()
print("Smokers (clinical.smoker = true):")
for r in rows2:
    print(f"  {r['full_name']:<22s}  smoker={r['smoker']}")

print()
# Filter on array length — conditions count
rows3 = conn.execute("""
    SELECT full_name,
           json_array_length(json_extract(clinical,'$.conditions')) AS n_conditions
    FROM   patients
    WHERE  json_array_length(json_extract(clinical,'$.conditions')) >= 2
    ORDER  BY n_conditions DESC
""").fetchall()
print("Patients with >= 2 conditions:")
for r in rows3:
    print(f"  {r['full_name']:<22s}  n_conditions={r['n_conditions']}")

print()
print("PostgreSQL JSONB filtering operators:")
pg_filters = [
    ("col @> '{\"province\":\"NB\"}'",     "Contains key-value pair (GIN indexable)"),
    ("col ? 'key'",                         "Key exists at top level"),
    ("col ?| ARRAY['k1','k2']",            "Any of these keys exist"),
    ("col ?& ARRAY['k1','k2']",            "All of these keys exist"),
    ("jsonb_array_length(col->'arr')",     "Length of a JSON array (PostgreSQL)"),
    ("col->>'age')::int > 50",             "Cast text to int for numeric comparison"),
]
for op, desc in pg_filters:
    print(f"  {op:<44s}  {desc}")


---
## JSON arrays: json_each and expansion

In [ ]:
print("=== JSON arrays: json_each and array indexing ===")
print()

# json_each — expand an array to rows (like UNNEST in PostgreSQL)
rows = conn.execute("""
    SELECT p.full_name,
           jc.value AS condition
    FROM   patients p,
           json_each(json_extract(p.clinical, '$.conditions')) AS jc
    ORDER  BY p.full_name, jc.value
""").fetchall()
print("json_each: expand conditions array to rows:")
print(f"  {'full_name':<22s}  condition")
print("  " + "-"*40)
for r in rows:
    print(f"  {r['full_name']:<22s}  {r['condition']}")

print()
# Count per condition across all patients
rows2 = conn.execute("""
    SELECT jc.value AS condition,
           COUNT(p.patient_id) AS n_patients
    FROM   patients p,
           json_each(json_extract(p.clinical, '$.conditions')) AS jc
    GROUP  BY jc.value
    ORDER  BY n_patients DESC
""").fetchall()
print("Condition prevalence across all patients:")
for r in rows2:
    print(f"  {r['condition']:<24s}  {r['n_patients']} patients")

print()
print("PostgreSQL equivalents for array operations:")
pg_array = [
    ("json_each(col)",          "jsonb_each(col)          -- expand object to key-value rows"),
    ("json_each_text(col)",     "jsonb_each_text(col)     -- same but values as text"),
    ("json_each on array",      "jsonb_array_elements(col->'arr') -- expand array to rows"),
    ("json_extract(c,'$[0]')",  "col->'arr'->0            -- first element"),
    ("json_array_length(arr)",  "jsonb_array_length(col->'arr') -- array length"),
    ("json_each + GROUP BY",    "jsonb_array_elements_text + GROUP BY -- same pattern"),
]
for sqlite, pg in pg_array:
    print(f"  {sqlite:<28s}  {pg}")


---
## Common Pitfalls

**1. Storing JSON as TEXT without validation**
In SQLite, `JSON` is just a type affinity alias for TEXT — any string can be stored, valid or not. `INSERT INTO patients (demographics) VALUES ('not json')` succeeds. In PostgreSQL, `JSON` and `JSONB` columns reject invalid JSON on insert. In SQLite, validate with `json_valid(col)` in a CHECK constraint: `CHECK (json_valid(demographics))`.

**2. json_extract returning NULL for missing paths (not an error)**
`json_extract(col, '$.missing_key')` returns NULL silently — not an error. In WHERE clauses, `WHERE json_extract(col, '$.x') = 'value'` excludes rows where the path is missing (NULL != 'value'). Use `IS NULL` or `COALESCE` explicitly.

**3. Type coercion: everything comes back as TEXT from ->>**
`json_extract(demographics, '$.age')` in SQLite returns a Python int if the JSON value is a number, but PostgreSQL's `->>` always returns TEXT. Always cast explicitly in PostgreSQL: `(col->>'age')::int > 50`.

**4. Using JSON_TABLE (MySQL/SQL Server) — not available in SQLite/PostgreSQL**
`JSON_TABLE` is a MySQL/SQL Server extension. SQLite uses `json_each()`; PostgreSQL uses `jsonb_array_elements()` and `jsonb_each()`. These are not interchangeable — always check dialect before copying JSON SQL.

**5. No index on JSON columns by default**
`WHERE json_extract(col, '$.province') = 'NB'` on SQLite scans the entire table and evaluates `json_extract` for every row. In PostgreSQL, create an expression index: `CREATE INDEX ON patients ((demographics->>'province'))`. SQLite supports expression indexes too: `CREATE INDEX ON patients (json_extract(demographics,'$.province'))`.

**6. json_each returning 'key' as string index for arrays**
When `json_each()` iterates a JSON array (not object), the `key` column contains the integer index as a string: '0', '1', '2'. Confusing it with the `value` column (which contains the actual element) is a common mistake when writing array expansion queries.


---
*sql_methods_library - Samantha McGarrigle*